# BI Tool Reviews — Theme Analysis
Uses GPT-4o to discover 5 recurring themes across 880 reviews, then tags each review with its theme.

## 1 · Setup

In [ ]:
# Install dependencies if needed
# !pip install openai pandas openpyxl tqdm

In [ ]:
import os
import json
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# ── Set your OpenAI API key ────────────────────────────────────────────────────
client = OpenAI(api_key="enter your OpenAI API key here")   
MODEL  = "gpt-4o"

# ── Load the dataset ──────────────────────────────────────────────────────────
df= pd.read_excel(r"E:\Thesis-2025\Results\Userfriendliness\Merged_Reviews.xlsx")
print(f"Loaded {len(df):,} reviews | columns: {df.columns.tolist()}")

Loaded 880 reviews | columns: ['Source', 'Review date', 'Review title', 'Review text', 'Reviewer role', 'Reviewer industry']


## 2 · Step 1 — Discover the 5 recurring themes

We send a **random sample of 80 reviews** to GPT and ask it to identify the 5 most repetitive themes. Sampling keeps costs low; 80 reviews are more than enough for pattern detection.

In [50]:
SAMPLE_SIZE = 80

sample_texts = (
    df["Review text"]
    .dropna()
    .sample(SAMPLE_SIZE, random_state=42)
    .str.slice(0, 400)          # trim very long reviews to keep the prompt compact
    .tolist()
)

numbered_sample = "\n\n".join(f"[{i+1}] {t}" for i, t in enumerate(sample_texts))

discovery_prompt = f"""
You are an expert in Human-Computer Interaction (HCI), usability engineering,
and qualitative thematic analysis.
Below are user reviews of BI tools (Power BI, Tableau, Looker).

Your task is to identify exactly 5 recurring THEMES STRICTLY RELATED TO SOFTWARE USABILITY.
The themes MUST align with established usability concepts from ISO 9241 and HCI literature, such as:
- effectiveness
- efficiency
- learnability
- ease of use
- user satisfaction
- flexibility
- error prevention
- workflow support
- visualization clarity

IMPORTANT RULES:
1. Focus ONLY on usability-related experiences.
2. Ignore business features, pricing, integrations, or marketing discussions
   unless directly tied to usability.
3. Themes must be mutually exclusive and collectively exhaustive (MECE).
4. Prefer abstract usability dimensions over product-specific features.
5. Each theme should represent a major dimension of user interaction quality.
6. Select the 5 dimensions most strongly evidenced by the reviews below,
   not necessarily all from the list above. The list is guidance, not a constraint.

Return ONLY valid JSON.
Prefer a top-level object with a "themes" list, for example:
{{"themes": [{{ "id": 1, "name": "...", "description": "..." }}]}}
If you return a bare array instead, it must be a list of theme objects.

REVIEWS:
{numbered_sample}
"""
 

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": discovery_prompt}],
    temperature=0.2,
    response_format={"type": "json_object"},
)

raw = response.choices[0].message.content
try:
    parsed = json.loads(raw)
except json.JSONDecodeError as err:
    raise ValueError(f"Expected JSON output from the model, but got:\n{raw}") from err

if isinstance(parsed, dict) and "themes" in parsed:
    themes = parsed["themes"]
elif isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
    themes = parsed
else:
    raise ValueError(
        f"Unexpected theme response format: {type(parsed).__name__} {parsed!r}\n"
        "Expected a list of theme objects or a JSON object with a top-level 'themes' list."
    )

if not isinstance(themes, list):
    raise ValueError(f"Expected themes to be a list, got {type(themes).__name__}: {themes!r}")

print("Discovered themes:\n")
for t in themes:
    print(f"  {t['id']}. {t['name']} — {t['description']}")

Discovered themes:

  1. Learnability — Users frequently mention the steep learning curve associated with BI tools, indicating challenges in mastering the software quickly. However, once learned, users find the tools powerful and effective.
  2. Ease of Use — Many reviews highlight the ease of use of the BI tools, particularly in terms of intuitive interfaces and drag-and-drop features, which facilitate user interaction and make the tools accessible to novices.
  3. Visualization Clarity — Users appreciate the ability to create clear and compelling visualizations, which aid in data analysis and decision-making. The tools are praised for their visualization capabilities, which help users understand complex data.
  4. Efficiency — The tools are noted for their efficiency in automating reports and reducing the time required for data analysis. Users find that these tools streamline workflows and enhance productivity.
  5. User Satisfaction — Overall user satisfaction is high, with many use

## 3 · Step 2 — Assign a theme to every review

We process reviews in **batches of 20** to balance speed and cost. Each batch is a single GPT call.

In [51]:
# Build a compact reference string of the 5 themes for the prompt
theme_ref = "\n".join(f"{t['id']}. {t['name']}: {t['description']}" for t in themes)

def assign_themes_batch(batch_texts: list[str]) -> list[str]:
    """
    Send a batch of review texts to GPT.
    Returns a list of theme names (same length as batch_texts).
    """
    numbered = "\n\n".join(f"[{i}] {str(t)[:400]}" for i, t in enumerate(batch_texts))

    prompt = f"""
You are a text classifier. Assign each review to the single best-matching theme.

THEMES:
{theme_ref}

REVIEWS (indexed 0 to {len(batch_texts)-1}):
{numbered}

Return ONLY a JSON object mapping each index (as a string) to the theme NAME.
Example: {{"0": "Ease of Use", "1": "Data Integration", ...}}
"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        response_format={"type": "json_object"},
    )

    mapping = json.loads(resp.choices[0].message.content)
    return [mapping.get(str(i), "Unknown") for i in range(len(batch_texts))]


# ── Run over the full dataset in batches ──────────────────────────────────────
BATCH_SIZE = 20
reviews    = df["Review text"].fillna("").tolist()
all_themes = []

for start in tqdm(range(0, len(reviews), BATCH_SIZE), desc="Classifying"):
    batch  = reviews[start : start + BATCH_SIZE]
    result = assign_themes_batch(batch)
    all_themes.extend(result)

df["Theme"] = all_themes
print("\nTheme distribution:")
print(df["Theme"].value_counts())

Classifying: 100%|██████████| 44/44 [01:44<00:00,  2.37s/it]


Theme distribution:
Theme
Ease of Use              280
Visualization Clarity    187
Learnability             164
User Satisfaction        126
Efficiency               121
Unknown                    2
Name: count, dtype: int64


In [52]:
df

,Source,Review date,Review title,Review text,Reviewer role,Reviewer industry,Theme
0,Looker,2026-01-19,Looker Review,Powerful BI and analytics platform that unifie...,Project Manager,Design,Learnability
1,Looker,2025-10-01,Great software for productivity tracking.,We are using it in our data extraction pipelin...,Sr. Data Analyst,Information Technology and Services,Efficiency
2,Looker,2025-09-26,"A good BI tool for quick solutions, but with l...",I found that blending data from multiple data ...,Data Visualization Engineer,Industrial Automation,Visualization Clarity
3,Looker,2025-08-09,Powerful tool that empowers data ownership,It has blended data feature which allows to cr...,Data Analyst,Information Technology and Services,Efficiency
4,Looker,2025-08-07,Solid dashboard tool,It is very easy to configure and implement das...,AI Data and Insights Analyst,Information Technology and Services,Ease of Use
...,...,...,...,...,...,...,...
875,Power BI,2021-06-03,How Powerful is Power BI ?,I am a Power BI Developer and have delivered m...,Power BI Developer,Computer Software,Ease of Use
876,Power BI,2021-05-04,Power BI ; Affordable Data Visualization Tool,It is an affordable software that enables us t...,Charity Njogu,Real Estate,Visualization Clarity
877,Power BI,2019-08-26,It brings the power of BI down to non-tech users,My job is to collect and process all sorts of ...,Senior Executive Officer,Banking,Visualization Clarity
878,Power BI,2017-09-22,Power BI is one of the most user friendly BI s...,Overall good software for beginners and can be...,Senior Manager Finance,Public Relations and Communications,Efficiency


In [54]:
df['Review text'][1]

'We are using it in our data extraction pipeline to track the time analysts spend on each document and their activities. This helps us set realistic targets for the team and boosts overall productivity. We also share daily reports with the team, which has further improved performance and efficiency, We appreciated Looker’s wide range of features, but what impressed us the most was its ability to create productivity metrics and provide consistent time and activity tracking., Integration with other software is a challenging task for the development team. Looker should provide a clear and user-friendly integration guide to make the process easier.'

## 4 · Preview & Save

In [55]:
# Preview
df[["Source", "Review title", "Theme"]].head(10)

,Source,Review title,Theme
0,Looker,Looker Review,Learnability
1,Looker,Great software for productivity tracking.,Efficiency
2,Looker,"A good BI tool for quick solutions, but with l...",Visualization Clarity
3,Looker,Powerful tool that empowers data ownership,Efficiency
4,Looker,Solid dashboard tool,Ease of Use
5,Looker,A Rich Tool Sometimes Overlooked Compared to C...,Ease of Use
6,Looker,Versatile and one size solution for data analy...,Ease of Use
7,Looker,My go-to for client reports,Learnability
8,Looker,Looker is a solid BI tool,User Satisfaction
9,Looker,Great dashboard tool,Visualization Clarity


In [56]:
# Save the enriched dataset
output_path = "Merged_Reviews_Themed.xlsx"
df.to_excel(output_path, index=False)
print(f"Saved → {output_path}")

Exception ignored in: <function tqdm.__del__ at 0x0000024A3E8B7EC0>
Traceback (most recent call last):
  File "C:\Users\siava\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "C:\Users\siava\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\notebook.py", line 277, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


Saved → Merged_Reviews_Themed.xlsx


## 5 · Quick breakdown by Source × Theme

In [62]:
pivot = df.pivot_table(index="Theme", columns="Source", aggfunc="size", fill_value=0)
pivot["Total"] = pivot.sum(axis=1)
pivot.sort_values("Total", ascending=False)

Source,Looker,Power BI,Tableau,Total
Theme,,,,
Ease of Use,102,81,97,280
Visualization Clarity,38,64,85,187
Learnability,56,71,37,164
User Satisfaction,39,50,37,126
Efficiency,43,34,44,121
Unknown,2,0,0,2
